<a href="https://colab.research.google.com/github/samueltsaii/Personal_Projects/blob/main/Semantic_Liver_Mass_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cv2
import json

In [ ]:
img = cv2.imread("drive/MyDrive/data/Benign/Benign/image/1.jpg")
if img is None:
    print("Error: Image not found")
else:
    img_0 = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img_0)
    plt.title("First image from data set")
    plt.axis("off")
    plt.show()


In [ ]:
base_dir = "drive/MyDrive/data"

categories = ["Benign", "Malignant"]

data = []

for category in categories:
    image_dir = os.path.join(base_dir, category, category, "image")
    seg_dir = os.path.join(base_dir, category, category, "segmentation", "mass")

    image_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(("png", "jpg", "jpeg"))])
    json_files = sorted([f for f in os.listdir(seg_dir) if f.lower().endswith("json")])

    for img, jsn in zip(image_files, json_files):
        data.append({
            "Category":category,
            "Image_Path":os.path.join(image_dir, img),
            "Segmentation_JSON_Path": os.path.join(seg_dir, jsn)
        })

df = pd.DataFrame(data)

In [ ]:
df.head()

In [ ]:
df.loc[0]["Segmentation_JSON_Path"]

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:
counts = df['Category'].value_counts()

categories = counts.index
frequencies = counts.values

plt.figure(figsize=(8, 5))

colors = ['red' if cat == 'Malignant' else 'green' for cat in categories]

plt.bar(categories, frequencies, color=colors)

plt.xlabel("Mass Categories")

plt.ylabel("Frequency")
plt.title("Frequency per Mass Categories")

plt.show()

In [ ]:
def generate_mask_from_json(json_path, image_shape):
    with open(json_path, 'r') as f:
        points_list = json.load(f)

    mask = np.zeros(image_shape[:2], dtype=np.uint8)

    points = np.array(points_list, dtype=np.int32)

    if len(points) >= 3:
        cv2.fillPoly(mask, [points], 255)

    return mask

def display_images_with_masks(df):
    categories = df['Category'].unique()
    fig, axes = plt.subplots(len(categories), 5, figsize=(20, 10))
    fig.suptitle("5 Randomly Drawn Liver Image Samples.\n {Green: Benign, Red: Malignant}", fontsize=20)

    for row_idx, category in enumerate(categories):

        samples = df[df['Category'] == category].sample(5, random_state=42).reset_index(drop=True)

        for col_idx, row in samples.iterrows():
            img = cv2.imread(row['Image_Path'])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            try:
                mask = generate_mask_from_json(row['Segmentation_JSON_Path'], img.shape)
            except Exception as e:
                print(f"Error reading mask for {row['Segmentation_JSON_Path']}: {e}")
                mask = np.zeros(img.shape[:2], dtype=np.uint8)

            overlay = img.copy()
            alpha = 0.65
            beta = 0.35
            colour = [255, 0, 0] if 'Malignant' in row["Category"] else [0, 255, 0]
            overlay[mask == 255] = colour
            transparent_image = cv2.addWeighted(img, alpha, overlay, beta, 0)

            axes[row_idx, col_idx].imshow(transparent_image)
            axes[row_idx, col_idx].axis('off')

            if col_idx == 0:
                axes[row_idx, col_idx].set_ylabel(category, fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()

display_images_with_masks(df)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms.v2 as transforms
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Unet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        # Encoder
        self.e11 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.e12 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.e21 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.e22 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.e31 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.e32 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.e41 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.e42 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.e51 = nn.Conv2d(512, 1024, kernel_size=3, padding=1)
        self.e52 = nn.Conv2d(1024, 1024, kernel_size=3, padding=1)

        # Decoder
        self.upconv1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.d11 = nn.Conv2d(1024, 512, kernel_size=3, padding=1) # 512 (up) + 512 (skip)
        self.d12 = nn.Conv2d(512, 512, kernel_size=3, padding=1)

        self.upconv2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.d21 = nn.Conv2d(512, 256, kernel_size=3, padding=1) # 256 + 256
        self.d22 = nn.Conv2d(256, 256, kernel_size=3, padding=1)

        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.d31 = nn.Conv2d(256, 128, kernel_size=3, padding=1) # 128 + 128
        self.d32 = nn.Conv2d(128, 128, kernel_size=3, padding=1)

        self.upconv4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.d41 = nn.Conv2d(128, 64, kernel_size=3, padding=1) # 64 + 64
        self.d42 = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        self.outconv = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        xe1 = F.relu(self.e12(F.relu(self.e11(x))))
        xp1 = self.pool1(xe1)

        xe2 = F.relu(self.e22(F.relu(self.e21(xp1))))
        xp2 = self.pool2(xe2)

        xe3 = F.relu(self.e32(F.relu(self.e31(xp2))))
        xp3 = self.pool3(xe3)

        xe4 = F.relu(self.e42(F.relu(self.e41(xp3))))
        xp4 = self.pool4(xe4)

        xe5 = F.relu(self.e52(F.relu(self.e51(xp4))))

        # Decoder
        xu1 = self.upconv1(xe5)
        xu1 = torch.cat([xu1, xe4], dim=1)
        xd1 = F.relu(self.d12(F.relu(self.d11(xu1))))

        xu2 = self.upconv2(xd1)
        xu2 = torch.cat([xu2, xe3], dim=1)
        xd2 = F.relu(self.d22(F.relu(self.d21(xu2))))

        xu3 = self.upconv3(xd2)
        xu3 = torch.cat([xu3, xe2], dim=1)
        xd3 = F.relu(self.d32(F.relu(self.d31(xu3))))

        xu4 = self.upconv4(xd3)
        xu4 = torch.cat([xu4, xe1], dim=1)
        xd4 = F.relu(self.d42(F.relu(self.d41(xu4))))

        return self.outconv(xd4)

In [ ]:
class LiverSegmentationDataset(Dataset):
    def __init__(self, dataframe, image_key, annotation_key, transform = None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.images = dataframe[image_key]
        self.annotations = dataframe[annotation_key]
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = generate_mask_from_json(self.annotations[idx], image.shape)

        image = Image.fromarray(image)
        mask = Image.fromarray(mask)

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        return image, mask

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self):
        super(DiceBCELoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):
        inputs = F.softmax(inputs, dim=1)
        inputs = inputs[:, 1, :, :].reshape(-1)
        targets = targets.reshape(-1).float()

        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)
        BCE = F.binary_cross_entropy(inputs, targets, reduction='mean')

        return BCE + dice_loss

In [ ]:
transform = transforms.Compose([
    transforms.Resize((512,512)),
    transforms.ToTensor()
])

data_set = LiverSegmentationDataset(df, "Image_Path", "Segmentation_JSON_Path", transform)
img, mask = data_set[0]
print(f"Mask unique values: {torch.unique(mask)}")

In [ ]:
train_size = int(0.8 * len(data_set))
valid_size = len(data_set) - train_size

train_data_set, valid_data_set = random_split(data_set, [train_size, valid_size])

In [ ]:
train_loader = DataLoader(train_data_set, batch_size = 32, shuffle = True)
valid_loader = DataLoader(valid_data_set, batch_size = 32, shuffle = False)

In [ ]:
#Instantiating hyperparameters
model = Unet(n_classes = 2)
Epochs = 100
learning_rate = 1e-4
optimizer = optim.Adam(model.parameters(), lr= learning_rate)
criterion = CrossEntropyLoss()

In [ ]:
Training Loop:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model.to(device)
train_losses, valid_losses = [], []

for Epoch in range(Epochs):
    model.train()
    train_running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        labels = labels.squeeze(1).to(device=device, dtype=torch.long)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_running_loss += loss.item() * images.size(0)

    epoch_train_loss = train_running_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    model.eval()
    all_labels = []
    all_probs = []
    valid_running_loss = 0.0
    valid_running_dice = 0.0
    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.squeeze(1).to(device=device, dtype=torch.long)

            outputs = model(images)

            probs = F.softmax(outputs, dim = 1)
            liver_probs = probs[:, 1, :, :]

            batch_dice = dice_coef(outputs, labels)
            valid_running_dice += batch_dice.item() * images.size(0)

            all_probs.append(liver_probs.cpu().numpy().flatten())
            all_labels.append(labels.cpu().numpy().flatten())

            loss = criterion(outputs, labels)
            valid_running_loss += loss.item() * images.size(0)

    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)
    auc_score = roc_auc_score(all_labels, all_probs)

    epoch_valid_loss = valid_running_loss / len(valid_loader.dataset)
    epoch_valid_dice = valid_running_dice / len(valid_loader.dataset)
    valid_losses.append(epoch_valid_loss)

    print(f"ROC-AUC Score: {auc_score:.4f}")
    print(f'Epoch {Epoch + 1}/{Epochs} | Train Loss: {epoch_train_loss:.4f} | Valid Loss: {epoch_valid_loss:.4f}')
    print(f'Val Dice: {epoch_valid_dice:.4f} | Val AUC: {auc_score:.4f}')
    print('-' * 20)

In [ ]:
def visualize_results(model, dataloader, device, num_samples=3):
    model.eval()
    images, labels = next(iter(dataloader))

    images = images.to(device)
    labels = labels.to(device)

    with torch.no_grad():
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

    plt.figure(figsize=(12, 4 * num_samples))

    for i in range(num_samples):
        plt.subplot(num_samples, 3, i*3 + 1)
        plt.imshow(images[i].cpu().permute(1, 2, 0))
        plt.title("Original Image")
        plt.axis('off')

        plt.subplot(num_samples, 3, i*3 + 2)
        plt.imshow(labels[i].cpu().squeeze(), cmap='gray')
        plt.title("Ground Truth Mask")
        plt.axis('off')

        plt.subplot(num_samples, 3, i*3 + 3)
        plt.imshow(preds[i].cpu(), cmap='Purples', alpha=0.5)
        plt.title("Predicted Mask")
        plt.axis('off')

    plt.tight_layout()
    plt.savefig('sample_predictions.png')

visualize_results(model, valid_loader, device)